In [1]:
# ! uv pip install google
! uv pip install google-generativeai

Using Python 3.12.3 environment at: /home/rayudu/otherwork_assignment_all_jobs/Todo/ai_multi_agent_shopping_assistant/.venv
Checked 1 package in 3ms


In [1]:
import openai
from qdrant_client import QdrantClient

2026-06-25 18:36:19.393085936 [W:onnxruntime:Default, device_discovery.cc:164 DiscoverDevicesForPlatform] GPU device discovery failed: device_discovery.cc:89 ReadFileContents Failed to open file: "/sys/class/drm/card0/device/vendor"


In [2]:
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct

import pandas as pd
import openai
# Instead of: import openai
import google.generativeai as genai
from google import genai
from google.genai import types
from dotenv import load_dotenv
import os

2026-07-08 13:13:32.424040602 [W:onnxruntime:Default, device_discovery.cc:164 DiscoverDevicesForPlatform] GPU device discovery failed: device_discovery.cc:89 ReadFileContents Failed to open file: "/sys/class/drm/card0/device/vendor"
/tmp/ipykernel_962811/3916736685.py:7: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


### Embedding function

In [3]:
# Load API key
load_dotenv()
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
if not GOOGLE_API_KEY:
    raise ValueError("GOOGLE_API_KEY not found in .env file!")

# Initialize client
gemini_client = genai.Client(api_key=GOOGLE_API_KEY)

In [4]:
# 3. Define the embedding function with custom dimensionality
def get_embedding(text, model="gemini-embedding-001",):
    """
    Generates an embedding vector for the given text using Gemini's current model.
    """
    response = gemini_client.models.embed_content(
        model="gemini-embedding-001", 
        contents=text,
        config=types.EmbedContentConfig(
        #     task_type=task_type
        #     # 🔴 THE FIX: Force the output to be exactly 1536 dimensions
        #     # output_dimensionality=1536  
        )
    )
    
    return response.embeddings[0].values

### Retrieval function

In [5]:
qdrant_client = QdrantClient(url="http://localhost:6333")

/tmp/ipykernel_962811/3504550968.py:1: UserWarning: Qdrant client version 1.16.2 is incompatible with server version 1.18.2. Major versions should match and minor version difference must not exceed 1. Set check_compatibility=False to skip version check.
  qdrant_client = QdrantClient(url="http://localhost:6333")


In [6]:
def retrieve_data(query, qdrant_client, k=5):

    query_embedding = get_embedding(query)

    results = qdrant_client.query_points(
        collection_name="Amazon-items-collection-03",
        query=query_embedding,
        limit=k,
    )

    retrieved_context_ids = []
    retrieved_context = []
    similarity_scores = []
    retrieved_context_ratings = []

    for result in results.points:
        retrieved_context_ids.append(result.payload["parent_asin"])
        retrieved_context.append(result.payload["description"])
        retrieved_context_ratings.append(result.payload["average_rating"])
        similarity_scores.append(result.score)

    return {
        "retrieved_context_ids": retrieved_context_ids,
        "retrieved_context": retrieved_context,
        "retrieved_context_ratings": retrieved_context_ratings,
        "similarity_scores": similarity_scores,
    }

In [7]:
retrieved_context = retrieve_data("What kind of earphones can I get?", qdrant_client, k=10)

In [8]:
retrieved_context

{'retrieved_context_ids': ['B0B67ZFRPC',
  'B0B9FTVL58',
  'B0C6K1GQCF',
  'B0B14HTZ59',
  'B09WCFC5D9',
  'B0BNHVLF7G',
  'B0CBMPG524',
  'B0C142QS8X',
  'B0B7495RL6',
  'B0BS15TRJ3'],
 'retrieved_context': ['QearFun Cat Earbuds for Kids, Kawakii Wired Earbud & in-Ear Headphones Gift for School Girls and Boys with Microphone and Lovely Earphones Storage Case(Blue) ',
  'Wireless Earbuds, Bluetooth 5.3 Headphones with Microphone, 37H Playback LED Power Display, In-Ear Headphones Deep Bass, IPX7 Waterproof, Ultra-Light Earphones with Charging Case, Smart Touch, Sport S23-vine earbuds',
  "TELSOR Wireless Earbuds for iPhone, Bluetooth Headphones Touch Control Stereo Sound Bluetooth Earbuds with Noise Cancelling Mic for Calls, 30H Playtime, IPX7 Waterproof Earbuds for Android, Black ♬【Bluetooth】Pair instantly with an uninterrupted and stable transmission with Bluetooth 5.1. AVRCP, HCP, HSP, and A2DP profiles are supported. The wireless earbuds are compatible with most Bluetooth enabled iP

### Format retrieved context function

In [9]:
def process_context(context):

    formatted_context = ""

    for id, chunk, rating in zip(context["retrieved_context_ids"], context["retrieved_context"], context["retrieved_context_ratings"]):
        formatted_context += f"- ID: {id}, rating: {rating}, description: {chunk}\n"

    return formatted_context

In [10]:
preprocessed_context = process_context(retrieved_context)

In [11]:
print(preprocessed_context)

- ID: B0B67ZFRPC, rating: 3.7, description: QearFun Cat Earbuds for Kids, Kawakii Wired Earbud & in-Ear Headphones Gift for School Girls and Boys with Microphone and Lovely Earphones Storage Case(Blue) 
- ID: B0B9FTVL58, rating: 4.2, description: Wireless Earbuds, Bluetooth 5.3 Headphones with Microphone, 37H Playback LED Power Display, In-Ear Headphones Deep Bass, IPX7 Waterproof, Ultra-Light Earphones with Charging Case, Smart Touch, Sport S23-vine earbuds
- ID: B0C6K1GQCF, rating: 4.3, description: TELSOR Wireless Earbuds for iPhone, Bluetooth Headphones Touch Control Stereo Sound Bluetooth Earbuds with Noise Cancelling Mic for Calls, 30H Playtime, IPX7 Waterproof Earbuds for Android, Black ♬【Bluetooth】Pair instantly with an uninterrupted and stable transmission with Bluetooth 5.1. AVRCP, HCP, HSP, and A2DP profiles are supported. The wireless earbuds are compatible with most Bluetooth enabled iPhones, Andriods, smart TVs, computers, etc. Each wireless earbuds will pair with each ot

### Create Prompt function

In [12]:
def build_prompt(preprocessed_context, question):

    prompt = f"""
You are a shopping assistant that can answer questions about the products in stock.

You will be given a question and a list of context.

Instructtions:
- You need to answer the question based on the provided context only.
- Never use word context and refer to it as the available products.

Context:
{preprocessed_context}

Question:
{question}
"""

    return prompt

In [13]:
prompt = build_prompt(preprocessed_context, "What kind of earphones can I get?")

In [14]:
print(prompt)


You are a shopping assistant that can answer questions about the products in stock.

You will be given a question and a list of context.

Instructtions:
- You need to answer the question based on the provided context only.
- Never use word context and refer to it as the available products.

Context:
- ID: B0B67ZFRPC, rating: 3.7, description: QearFun Cat Earbuds for Kids, Kawakii Wired Earbud & in-Ear Headphones Gift for School Girls and Boys with Microphone and Lovely Earphones Storage Case(Blue) 
- ID: B0B9FTVL58, rating: 4.2, description: Wireless Earbuds, Bluetooth 5.3 Headphones with Microphone, 37H Playback LED Power Display, In-Ear Headphones Deep Bass, IPX7 Waterproof, Ultra-Light Earphones with Charging Case, Smart Touch, Sport S23-vine earbuds
- ID: B0C6K1GQCF, rating: 4.3, description: TELSOR Wireless Earbuds for iPhone, Bluetooth Headphones Touch Control Stereo Sound Bluetooth Earbuds with Noise Cancelling Mic for Calls, 30H Playtime, IPX7 Waterproof Earbuds for Android, B

### Generate answer function

In [15]:
# AVAILABLE MODELS FOR gemini_client.models.generate_content:
# -----------------------------------------------------------
# "gemini-2.5-flash"        : Best for speed, low latency, and general tasks.
# "gemini-2.0-flash"        : High-speed, versatile, supports multi-modal input.
# "gemini-2.0-pro-exp-02-05": High performance, complex reasoning, and deep analysis.
# "gemini-2.0-flash-thinking-exp": Specialized for reasoning, math, and code logic.
# "gemini-1.5-flash"        : Legacy fast model, still widely used.
# "gemini-1.5-pro"          : Deep context understanding and complex multi-step reasoning.
# -----------------------------------------------------------

def generate_answer(prompt, model_name="gemini-2.5-flash"):
    """
    Generates a response using the specified Gemini model.
    """
    response = gemini_client.models.generate_content(
        model=model_name,
        contents=prompt,
        # You can add config here if you need system instructions or temperature control
    )
    
    return response.text

In [16]:
print(generate_answer(prompt))

Based on the available products, you can get the following types of earphones:

*   **QearFun Cat Earbuds for Kids (Blue)**: Kawakii wired earbuds designed for kids, featuring a microphone and a lovely storage case.
*   **Wireless Earbuds (S23-vine)**: Bluetooth 5.3 headphones with a microphone, offering 37 hours of playback, LED power display, deep bass, IPX7 waterproof rating, and an ultra-light design with a charging case and smart touch controls.
*   **TELSOR Wireless Earbuds (Black)**: Bluetooth 5.1 headphones with touch control, stereo sound, noise-cancelling mic for calls, 30 hours of playtime, and an IPX7 waterproof rating. They are compatible with most Bluetooth-enabled iPhones, Androids, smart TVs, and computers.
*   **8 Pairs Powerbeats Pro Ear Tips Buds (White)**: Replacement ear tips made of soft, flexible silicone in four sizes (Small, Medium, Large, Double Flange) for Powerbeats Pro, Powerbeats High Performance, Beats Flex, Beats X, Beats Fit Pro, and Beats Studio Buds. 

### Combined RAG Pipeline

In [17]:
def rag_pipeline(question, top_k=5):

    qdrant_client = QdrantClient(url="http://localhost:6333")

    retrieved_context = retrieve_data(question, qdrant_client, top_k)
    preprocessed_context = process_context(retrieved_context)
    prompt = build_prompt(preprocessed_context, question)
    answer = generate_answer(prompt)

    return answer

In [18]:
print(rag_pipeline("What kind of earphones can I get with ratings above 4.3?"))

/tmp/ipykernel_962811/2600847150.py:3: UserWarning: Qdrant client version 1.16.2 is incompatible with server version 1.18.2. Major versions should match and minor version difference must not exceed 1. Set check_compatibility=False to skip version check.
  qdrant_client = QdrantClient(url="http://localhost:6333")


Based on the available products, here are the earphones you can get with ratings above 4.3:

*   TELSOR Wireless Earbuds for iPhone, Bluetooth Headphones Touch Control Stereo Sound Bluetooth Earbuds with Noise Cancelling Mic for Calls, 30H Playtime, IPX7 Waterproof Earbuds for Android, Black. These earbuds feature Bluetooth 5.1 for stable transmission, 10mm speakers with 2 microphones for clear calls and deep bass, up to 30 hours of playtime with the charging case, IPX7 waterproof rating, and smart touch control.
